# Exploring a Dense Channel Dataset (MNE Sample)

**Topic**: High-resolution topographic maps using 60 channels from MNE sample dataset

---

## Overview

The previous example showed that 4 channels are insufficient for accurate topomaps. Here we explore the MNE sample dataset with 60 channels and full electrode montage.

**Data**: MNE sample dataset, 60 EEG channels, auditory/visual experiment, pre-configured 10-20 montage

## 1. Install dependencies

In [ ]:
!pip install mne matplotlib

## 2. Download dataset and load data

The MNE sample dataset downloads automatically via a single function. Size is about 1.5 GB (one-time download).

In [ ]:
import mne
import matplotlib.pyplot as plt

# Download the MNE sample dataset (1.5 GB, one-time download)
sample_path = mne.datasets.sample.data_path()

# Load the EEG data (60 channels, pre-configured montage)
raw_fname = sample_path / 'MEG' / 'sample' / 'sample_audvis_raw.fif'
raw = mne.io.read_raw_fif(raw_fname, preload=True)

# Pick only EEG channels (exclude MEG and other channels)
raw.pick_types(eeg=True)

print(f'Number of EEG channels: {len(raw.ch_names)}')
print(f'Channel names: {raw.ch_names[:10]} ...')
print(f'Sampling rate: {raw.info["sfreq"]} Hz')
print(f'Duration: {raw.times[-1]:.1f} s')
print(f'Montage: {raw.get_montage()}')

# Create epochs around auditory events (event ID 1 = standard tone)
events = mne.find_events(raw, stim_channel='STI 014')
epochs = mne.Epochs(raw, events, event_id=1, tmin=-0.2, tmax=0.5,
                    preload=True)

# Compute evoked response (average across epochs)
evoked = epochs.average()

# Plot topomap at multiple time points
fig = evoked.plot_topomap(times=[0.0, 0.1, 0.2, 0.3],
                          ch_type='eeg',
                          time_unit='s',
                          size=2)
plt.tight_layout()
plt.show()

## 3. Step-by-step explanation

- `mne.datasets.sample.data_path()`: Downloads data and returns the path
- `read_raw_fif`: Reads the file in MNE's native FIF format
- `pick_types(eeg=True)`: Excludes MEG and other channels, keeps EEG only
- `find_events`: Searches for stimulus onset times in channel STI 014
- `Epochs`: Cuts the signal into segments around each stimulus (-0.2 to +0.5 s)
- `average()`: Computes the average across epochs to reveal consistent evoked activity
- `plot_topomap`: Draws the topographic map at specified time points

### What to look for
- The dramatic difference in map quality compared to 4 channels
- Distribution of activity across the scalp at different time points after stimulus
- Colors gradient from red (positive) to blue (negative)
- Spatial interpolation fills gaps between electrodes smoothly

## Summary

- Channel density determines topomap accuracy: 60 channels produce smooth, expressive maps
- The montage defines 3D electrode positions on the scalp
- Event-related potentials (ERP) produce cleaner maps than continuous signal
- The 10-20 system names channels by anatomical location: F (frontal), C (central), P (parietal), T (temporal), O (occipital)